# 第17章 高级安全 - 加密技术 (Encryption)

## Cambridge A-Level Computer Science (9618)

---

### 本节学习目标

| 知识点 | 英文术语 | 重要程度 |
|--------|----------|----------|
| 对称加密 | Symmetric Encryption | ★★★★★ |
| 非对称加密 | Asymmetric Encryption | ★★★★★ |
| 异或运算与加密 | XOR in Cryptography | ★★★★ |
| RSA算法原理 | RSA Algorithm | ★★★★ |
| 加密对比分析 | Encryption Comparison | ★★★★ |

---
## 1. 什么是加密？(What is Encryption?)

**加密 (Encryption)** 是将**明文 (Plaintext)** 通过某种算法转换为**密文 (Ciphertext)** 的过程。

**解密 (Decryption)** 则是将密文还原为明文的逆过程。

### 为什么我们需要加密？

想象你在微信上给朋友发一条消息。这条消息会经过你的路由器、你的ISP（网络运营商）、多个中间节点、对方的ISP、对方的路由器……在这条路径上的**任何一个节点**都有可能截获你的数据。如果没有加密，你的消息就像写在明信片上一样，任何经手的人都能看到。

加密的三个核心目标 (CIA Triad)：
- **机密性 (Confidentiality)**：只有授权的人能读懂数据
- **完整性 (Integrity)**：数据在传输中没有被篡改
- **认证性 (Authentication)**：确认发送者的身份

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

# 设置中文字体
plt.rcParams['font.sans-serif'] = ['WenQuanYi Zen Hei', 'Noto Sans CJK SC', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False

fig, ax = plt.subplots(1, 1, figsize=(14, 4))
ax.set_xlim(0, 14)
ax.set_ylim(0, 4)
ax.axis('off')
ax.set_title('加密与解密的基本流程 (Encryption & Decryption)', fontsize=16, fontweight='bold')

# 明文
rect1 = mpatches.FancyBboxPatch((0.5, 1.2), 2.2, 1.6, boxstyle='round,pad=0.2', 
                                  facecolor='#4CAF50', edgecolor='black', linewidth=2)
ax.add_patch(rect1)
ax.text(1.6, 2.0, 'Plaintext\n(明文)', ha='center', va='center', fontsize=12, color='white', fontweight='bold')

# 加密箭头
ax.annotate('', xy=(4.0, 2.0), xytext=(2.9, 2.0),
            arrowprops=dict(arrowstyle='->', color='#FF5722', lw=2.5))
ax.text(3.45, 2.5, 'Encrypt\n(加密)', ha='center', va='center', fontsize=10, color='#FF5722')

# 密钥(上方)
rect_key1 = mpatches.FancyBboxPatch((3.8, 3.0), 1.8, 0.8, boxstyle='round,pad=0.15',
                                     facecolor='#FFC107', edgecolor='black', linewidth=1.5)
ax.add_patch(rect_key1)
ax.text(4.7, 3.4, 'Key (密钥)', ha='center', va='center', fontsize=10, fontweight='bold')
ax.annotate('', xy=(4.7, 2.85), xytext=(4.7, 3.0),
            arrowprops=dict(arrowstyle='->', color='#FFC107', lw=2))

# 加密算法
rect2 = mpatches.FancyBboxPatch((4.0, 1.2), 2.5, 1.6, boxstyle='round,pad=0.2',
                                  facecolor='#FF5722', edgecolor='black', linewidth=2)
ax.add_patch(rect2)
ax.text(5.25, 2.0, 'Algorithm\n(算法)', ha='center', va='center', fontsize=12, color='white', fontweight='bold')

# 密文
ax.annotate('', xy=(7.8, 2.0), xytext=(6.7, 2.0),
            arrowprops=dict(arrowstyle='->', color='#9C27B0', lw=2.5))
rect3 = mpatches.FancyBboxPatch((7.8, 1.2), 2.2, 1.6, boxstyle='round,pad=0.2',
                                  facecolor='#9C27B0', edgecolor='black', linewidth=2)
ax.add_patch(rect3)
ax.text(8.9, 2.0, 'Ciphertext\n(密文)', ha='center', va='center', fontsize=12, color='white', fontweight='bold')

# 解密箭头
ax.annotate('', xy=(11.3, 2.0), xytext=(10.2, 2.0),
            arrowprops=dict(arrowstyle='->', color='#2196F3', lw=2.5))
ax.text(10.75, 2.5, 'Decrypt\n(解密)', ha='center', va='center', fontsize=10, color='#2196F3')

# 还原明文
rect4 = mpatches.FancyBboxPatch((11.3, 1.2), 2.2, 1.6, boxstyle='round,pad=0.2',
                                  facecolor='#4CAF50', edgecolor='black', linewidth=2)
ax.add_patch(rect4)
ax.text(12.4, 2.0, 'Plaintext\n(明文)', ha='center', va='center', fontsize=12, color='white', fontweight='bold')

plt.tight_layout()


---
## 2. 异或运算：加密的基础 (XOR: The Foundation of Encryption)

### 为什么异或(XOR)运算是加密的基础？

异或运算有一个极其重要的数学性质：**它是可逆的！**

```
A XOR B = C
C XOR B = A  （用同一个密钥B再做一次XOR，就还原了A）
```

这意味着：
1. **同一个操作既能加密又能解密** —— 实现简单
2. **输出的每一位都依赖于输入和密钥** —— 改变任何一位都会改变结果
3. **如果密钥是随机的，输出看起来也是完全随机的** —— 无法从密文推断明文

AES、DES等现代加密算法的核心运算之一就是XOR。可以说，没有XOR就没有现代密码学。

| A | B | A XOR B |
|---|---|--------|
| 0 | 0 | 0 |
| 0 | 1 | 1 |
| 1 | 0 | 1 |
| 1 | 1 | 0 |

In [ ]:
# XOR的可逆性演示
print('='*60)
print('XOR 可逆性演示 (XOR Reversibility Demo)')
print('='*60)

plaintext = 0b11010110  # 明文: 214
key       = 0b10101011  # 密钥: 171

# 加密
ciphertext = plaintext ^ key

# 解密 (用同一个密钥再XOR一次)
decrypted = ciphertext ^ key

print(f'明文 (Plaintext):   {plaintext:08b}  = {plaintext}')
print(f'密钥 (Key):         {key:08b}  = {key}')
print(f'密文 (Ciphertext):  {ciphertext:08b}  = {ciphertext}')
print(f'解密 (Decrypted):   {decrypted:08b}  = {decrypted}')
print()
print(f'明文 == 解密结果?   {plaintext == decrypted}  <- 完美还原!')
print()
print('--- 逐位分析 (Bit-by-bit Analysis) ---')
print(f'  明文:  {plaintext:08b}')
print(f'  密钥:  {key:08b}')
print(f'  XOR:   {ciphertext:08b}  <- 密文')
print(f'  密钥:  {key:08b}')


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

plt.rcParams['font.sans-serif'] = ['WenQuanYi Zen Hei', 'Noto Sans CJK SC', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False

# 可视化：XOR加密如何让数据看起来随机
np.random.seed(42)

# 创建一个有规律的信号（明文）
x = np.arange(256)
plaintext_data = np.array([(int(np.sin(i/10)*50) + 100) % 256 for i in x], dtype=np.uint8)

# 随机密钥流
key_stream = np.random.randint(0, 256, size=256, dtype=np.uint8)

# XOR加密
ciphertext_data = plaintext_data ^ key_stream

fig, axes = plt.subplots(3, 1, figsize=(14, 8))

axes[0].bar(x, plaintext_data, color='#4CAF50', width=1.0, alpha=0.8)
axes[0].set_title('明文数据 (Plaintext) - 有明显规律', fontsize=13, fontweight='bold')
axes[0].set_ylabel('字节值')
axes[0].set_ylim(0, 260)

axes[1].bar(x, key_stream, color='#FFC107', width=1.0, alpha=0.8)
axes[1].set_title('密钥流 (Key Stream) - 随机', fontsize=13, fontweight='bold')
axes[1].set_ylabel('字节值')
axes[1].set_ylim(0, 260)

axes[2].bar(x, ciphertext_data, color='#F44336', width=1.0, alpha=0.8)
axes[2].set_title('密文数据 (Ciphertext = Plaintext XOR Key) - 看起来完全随机!', fontsize=13, fontweight='bold')
axes[2].set_ylabel('字节值')
axes[2].set_ylim(0, 260)

for ax in axes:
    ax.set_xlim(0, 256)

plt.tight_layout()
plt.show()

print('关键观察: 即使明文有明显规律(正弦波), XOR加密后看起来完全随机!')


---
## 3. 对称加密 (Symmetric Encryption)

### 核心概念

对称加密使用**同一把密钥**进行加密和解密。就像一把钥匙既能锁门又能开门。

### 为什么叫"对称"？

因为加密和解密双方使用的是**同一个密钥**，双方处于"对称"的地位 —— 拥有同样的秘密。

### 经典算法

| 算法 | 密钥长度 | 特点 | 状态 |
|------|---------|------|------|
| DES (Data Encryption Standard) | 56位 | 历史意义大，但密钥太短 | 已不安全，被淘汰 |
| 3DES (Triple DES) | 168位 | DES执行三次 | 渐被淘汰 |
| AES (Advanced Encryption Standard) | 128/192/256位 | 当前标准 | 广泛使用 |

### AES 为什么是标准？

1. **安全性**：AES-256有2^256种可能的密钥，即使全世界的计算机一起暴力破解也需要数十亿年
2. **效率**：硬件可以高效实现AES运算，现代CPU甚至有专门的AES指令集
3. **灵活性**：支持128/192/256位三种密钥长度

### 对称加密的根本问题：密钥分发 (Key Distribution Problem)

如果你和朋友共用一个密码来加密信息，你们首先得**安全地**把这个密码传给对方。但如果你们已经有安全的通道，为什么还需要加密？这就是"密钥分发问题"。

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

plt.rcParams['font.sans-serif'] = ['WenQuanYi Zen Hei', 'Noto Sans CJK SC', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False

fig, ax = plt.subplots(1, 1, figsize=(14, 6))
ax.set_xlim(0, 14)
ax.set_ylim(0, 6)
ax.axis('off')
ax.set_title('对称加密流程 (Symmetric Encryption)', fontsize=16, fontweight='bold')

# Alice
circle_a = plt.Circle((1.5, 3), 0.8, color='#2196F3', ec='black', lw=2)
ax.add_patch(circle_a)
ax.text(1.5, 3, 'Alice\n(发送方)', ha='center', va='center', fontsize=10, color='white', fontweight='bold')

# 明文
rect1 = mpatches.FancyBboxPatch((2.8, 2.3), 1.6, 1.4, boxstyle='round,pad=0.15',
                                  facecolor='#4CAF50', edgecolor='black', linewidth=1.5)
ax.add_patch(rect1)
ax.text(3.6, 3.0, 'Hello!\n(明文)', ha='center', va='center', fontsize=10, color='white', fontweight='bold')

# 密钥(上方居中)
rect_key = mpatches.FancyBboxPatch((5.5, 4.8), 3.0, 0.8, boxstyle='round,pad=0.15',
                                    facecolor='#FFC107', edgecolor='black', linewidth=2)
ax.add_patch(rect_key)
ax.text(7.0, 5.2, 'Shared Key: K (共享密钥)', ha='center', va='center', fontsize=11, fontweight='bold')

# 密钥箭头到加密和解密
ax.annotate('', xy=(5.7, 3.6), xytext=(6.2, 4.8),
            arrowprops=dict(arrowstyle='->', color='#FFC107', lw=2, ls='--'))
ax.annotate('', xy=(8.3, 3.6), xytext=(7.8, 4.8),
            arrowprops=dict(arrowstyle='->', color='#FFC107', lw=2, ls='--'))

# 加密过程
ax.annotate('', xy=(5.0, 3.0), xytext=(4.5, 3.0),
            arrowprops=dict(arrowstyle='->', color='black', lw=2))
rect_enc = mpatches.FancyBboxPatch((5.0, 2.3), 1.5, 1.4, boxstyle='round,pad=0.15',
                                    facecolor='#FF5722', edgecolor='black', linewidth=1.5)
ax.add_patch(rect_enc)
ax.text(5.75, 3.0, 'Encrypt\n(加密)', ha='center', va='center', fontsize=10, color='white', fontweight='bold')

# 密文通过网络
ax.annotate('', xy=(7.8, 3.0), xytext=(6.6, 3.0),
            arrowprops=dict(arrowstyle='->', color='#9C27B0', lw=2.5))
ax.text(7.2, 2.0, '@#$%&\n(密文经网络传输)', ha='center', va='center', fontsize=9, color='#9C27B0')

# 解密过程
rect_dec = mpatches.FancyBboxPatch((7.8, 2.3), 1.5, 1.4, boxstyle='round,pad=0.15',
                                    facecolor='#2196F3', edgecolor='black', linewidth=1.5)
ax.add_patch(rect_dec)
ax.text(8.55, 3.0, 'Decrypt\n(解密)', ha='center', va='center', fontsize=10, color='white', fontweight='bold')

# 还原明文
ax.annotate('', xy=(10.5, 3.0), xytext=(9.4, 3.0),
            arrowprops=dict(arrowstyle='->', color='black', lw=2))
rect2 = mpatches.FancyBboxPatch((10.5, 2.3), 1.6, 1.4, boxstyle='round,pad=0.15',
                                  facecolor='#4CAF50', edgecolor='black', linewidth=1.5)
ax.add_patch(rect2)
ax.text(11.3, 3.0, 'Hello!\n(明文)', ha='center', va='center', fontsize=10, color='white', fontweight='bold')

# Bob
circle_b = plt.Circle((13.0, 3), 0.8, color='#E91E63', ec='black', lw=2)
ax.add_patch(circle_b)
ax.text(13.0, 3, 'Bob\n(接收方)', ha='center', va='center', fontsize=10, color='white', fontweight='bold')

# 标注重点
ax.text(7.0, 0.8, '关键问题: Alice和Bob必须先安全地共享密钥K!', ha='center', va='center',
        fontsize=12, color='red', fontweight='bold',
        bbox=dict(boxstyle='round,pad=0.3', facecolor='#FFEBEE', edgecolor='red', lw=1.5))

plt.tight_layout()


---
## 4. 凯撒密码 (Caesar Cipher) - 最简单的对称加密

凯撒密码是历史上最早的加密方法之一。虽然它极不安全（只有25种可能的密钥），但它完美地展示了对称加密的核心思想。

**原理**：将字母表中的每个字母向后移动固定的位数。

In [ ]:
def caesar_encrypt(plaintext, shift):
    """凯撒密码加密 (Caesar Cipher Encryption)"""
    result = []
    for char in plaintext:
        if char.isalpha():
            base = ord('A') if char.isupper() else ord('a')
            encrypted = chr((ord(char) - base + shift) % 26 + base)
            result.append(encrypted)
        else:
            result.append(char)
    return ''.join(result)

def caesar_decrypt(ciphertext, shift):
    """凯撒密码解密 (Caesar Cipher Decryption)"""
    return caesar_encrypt(ciphertext, -shift)  # 反向移位就是解密

# 演示
print('='*60)
print('凯撒密码演示 (Caesar Cipher Demo)')
print('='*60)

message = 'Hello World'
shift = 3

encrypted = caesar_encrypt(message, shift)
decrypted = caesar_decrypt(encrypted, shift)

print(f'原始消息 (Plaintext):  {message}')
print(f'密钥 (Key/Shift):      {shift}')
print(f'加密结果 (Ciphertext): {encrypted}')
print(f'解密结果 (Decrypted):  {decrypted}')
print()

# 展示所有可能的密钥（暴力破解）
print('--- 暴力破解凯撒密码 (Brute Force Attack) ---')
secret = caesar_encrypt('ATTACK AT DAWN', 7)
print(f'截获密文: {secret}')
print()
for s in range(1, 26):
    attempt = caesar_decrypt(secret, s)
    marker = ' <-- FOUND!' if 'ATTACK' in attempt else ''


---
## 5. XOR加密的Python实现

XOR加密比凯撒密码安全得多，因为密钥空间大大增加了。这是理解现代加密算法的基础。

In [ ]:
def xor_encrypt_decrypt(data, key):
    """XOR加密/解密（同一个函数，因为XOR是可逆的）"""
    result = bytearray()
    for i, byte in enumerate(data):
        # 密钥循环使用: key[i % len(key)]
        key_byte = key[i % len(key)]
        result.append(byte ^ key_byte)
    return bytes(result)

# 演示
print('='*60)
print('XOR加密演示 (XOR Encryption Demo)')
print('='*60)

message = 'Cambridge A-Level CS is awesome!'
key = 'SECRET'

# 转换为字节
msg_bytes = message.encode('utf-8')
key_bytes = key.encode('utf-8')

# 加密
encrypted = xor_encrypt_decrypt(msg_bytes, key_bytes)

# 解密（用同一个密钥和同一个函数！）
decrypted = xor_encrypt_decrypt(encrypted, key_bytes)

print(f'明文:   {message}')
print(f'密钥:   {key}')
print(f'密文 (hex): {encrypted.hex()}')
print(f'密文 (raw): {encrypted}')
print(f'解密:   {decrypted.decode("utf-8")}')
print()
print('注意: 加密和解密使用的是完全相同的函数和密钥!')
print('这就是XOR的美妙之处 -- A XOR B XOR B = A')

# 逐字节展示
print()
print('--- 逐字节分析 (前10个字节) ---')
print(f'{"明文字符":>8} {"明文byte":>8} {"密钥byte":>8} {"XOR结果":>8} {"密文hex":>8}')
for i in range(min(10, len(msg_bytes))):
    k = key_bytes[i % len(key_bytes)]
    c = msg_bytes[i] ^ k


---
## 6. 非对称加密 (Asymmetric Encryption)

### 核心概念

非对称加密使用**一对密钥**：
- **公钥 (Public Key)**：公开给所有人，用于加密
- **私钥 (Private Key)**：只有自己知道，用于解密

### 为什么叫"非对称"？

因为加密和解密使用**不同的密钥**，双方不再"对称"—— 一方有公钥，一方有私钥。

### 革命性的解决方案

非对称加密解决了对称加密的**密钥分发问题**：
- Bob把自己的公钥公开发布
- Alice用Bob的公钥加密消息
- 只有Bob的私钥能解密
- **即使有人截获了公钥和密文，也无法解密！**

### RSA算法 —— 最经典的非对称加密

RSA由Rivest、Shamir、Adleman三人于1977年发明。

### 大质数分解为什么难？这就是RSA安全的基础

RSA的安全性基于一个数学事实：

> **将两个大质数相乘很容易，但将乘积分解回两个质数极其困难。**

例如：
- 计算 `61 x 53 = 3233` → 一瞬间
- 已知 `3233`，求它的两个质因数？→ 需要逐个尝试

当数字增长到几百位时，现有计算机需要**数十亿年**才能完成分解。

这叫做**计算非对称性 (Computational Asymmetry)**——正向计算容易，逆向计算几乎不可能。

In [ ]:
import time
import random

print('='*60)
print('大质数分解的困难性演示')
print('(Why Prime Factorization is Hard)')
print('='*60)
print()

def is_prime(n):
    if n < 2: return False
    if n < 4: return True
    if n % 2 == 0 or n % 3 == 0: return False
    i = 5
    while i * i <= n:
        if n % i == 0 or n % (i + 2) == 0: return False
        i += 6
    return True

def find_factors(n):
    """暴力分解质因数，返回(因子, 尝试次数)"""
    attempts = 0
    for i in range(2, int(n**0.5) + 1):
        attempts += 1
        if n % i == 0:
            return (i, n // i), attempts
    return None, attempts

# 测试不同大小的质数乘积
test_cases = [
    (61, 53),         # 小质数
    (541, 761),       # 中等质数
    (7919, 7907),     # 较大质数
    (104729, 104723), # 大质数
]

print(f'{"质数p":>10} {"质数q":>10} {"乘积n":>15} {"分解尝试次数":>12} {"时间(ms)":>10}')
print('-' * 65)

for p, q in test_cases:
    n = p * q
    
    start = time.perf_counter()
    factors, attempts = find_factors(n)
    elapsed = (time.perf_counter() - start) * 1000
    
    print(f'{p:>10} {q:>10} {n:>15,} {attempts:>12,} {elapsed:>9.3f}')

print()
print('观察: 当质数变大时, 分解所需的尝试次数急剧增加!')
print('RSA使用的是300+位的质数, 其乘积有600+位,')


In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

plt.rcParams['font.sans-serif'] = ['WenQuanYi Zen Hei', 'Noto Sans CJK SC', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False

fig, ax = plt.subplots(1, 1, figsize=(14, 7))
ax.set_xlim(0, 14)
ax.set_ylim(0, 7)
ax.axis('off')
ax.set_title('非对称加密流程 (Asymmetric Encryption)', fontsize=16, fontweight='bold')

# Alice
circle_a = plt.Circle((1.5, 3.5), 0.9, color='#2196F3', ec='black', lw=2)
ax.add_patch(circle_a)
ax.text(1.5, 3.5, 'Alice\n(发送方)', ha='center', va='center', fontsize=10, color='white', fontweight='bold')

# Bob
circle_b = plt.Circle((12.5, 3.5), 0.9, color='#E91E63', ec='black', lw=2)
ax.add_patch(circle_b)
ax.text(12.5, 3.5, 'Bob\n(接收方)', ha='center', va='center', fontsize=10, color='white', fontweight='bold')

# Bob的密钥对
rect_pub = mpatches.FancyBboxPatch((9.8, 5.5), 2.2, 0.9, boxstyle='round,pad=0.15',
                                    facecolor='#4CAF50', edgecolor='black', linewidth=2)
ax.add_patch(rect_pub)
ax.text(10.9, 5.95, 'Public Key (公钥)', ha='center', va='center', fontsize=10, fontweight='bold', color='white')

rect_priv = mpatches.FancyBboxPatch((9.8, 1.0), 2.2, 0.9, boxstyle='round,pad=0.15',
                                     facecolor='#F44336', edgecolor='black', linewidth=2)
ax.add_patch(rect_priv)
ax.text(10.9, 1.45, 'Private Key (私钥)', ha='center', va='center', fontsize=10, fontweight='bold', color='white')

# 步骤1: Bob公开公钥
ax.annotate('', xy=(4.0, 5.95), xytext=(9.7, 5.95),
            arrowprops=dict(arrowstyle='->', color='#4CAF50', lw=2.5, ls='--'))
ax.text(6.8, 6.5, 'Step 1: Bob公开公钥', ha='center', va='center', fontsize=11,
        color='#4CAF50', fontweight='bold')

# Alice用公钥加密
rect_enc = mpatches.FancyBboxPatch((3.5, 2.8), 2.5, 1.4, boxstyle='round,pad=0.15',
                                    facecolor='#FF9800', edgecolor='black', linewidth=2)
ax.add_patch(rect_enc)
ax.text(4.75, 3.5, 'Encrypt with\nPublic Key\n(公钥加密)', ha='center', va='center', fontsize=9, fontweight='bold')

ax.annotate('', xy=(3.4, 3.5), xytext=(2.5, 3.5),
            arrowprops=dict(arrowstyle='->', color='black', lw=2))

# 步骤2标注
ax.text(4.75, 4.5, 'Step 2', ha='center', va='center', fontsize=10, color='#FF9800', fontweight='bold')

# 密文传输
ax.annotate('', xy=(9.0, 3.5), xytext=(6.2, 3.5),
            arrowprops=dict(arrowstyle='->', color='#9C27B0', lw=2.5))
ax.text(7.6, 4.0, 'Step 3: 密文通过网络', ha='center', va='center', fontsize=10,
        color='#9C27B0', fontweight='bold')
ax.text(7.6, 3.1, '@#$%&*!', ha='center', va='center', fontsize=12, color='#9C27B0')

# Bob用私钥解密
rect_dec = mpatches.FancyBboxPatch((9.0, 2.8), 2.5, 1.4, boxstyle='round,pad=0.15',
                                    facecolor='#673AB7', edgecolor='black', linewidth=2)
ax.add_patch(rect_dec)
ax.text(10.25, 3.5, 'Decrypt with\nPrivate Key\n(私钥解密)', ha='center', va='center', fontsize=9, color='white', fontweight='bold')

ax.text(10.25, 4.5, 'Step 4', ha='center', va='center', fontsize=10, color='#673AB7', fontweight='bold')

# 注释
ax.text(7.0, 0.4, 'Public Key可以公开 | Private Key必须保密 | 公钥加密的数据只有私钥能解密',
        ha='center', va='center', fontsize=11, fontweight='bold',
        bbox=dict(boxstyle='round,pad=0.4', facecolor='#E8F5E9', edgecolor='#4CAF50', lw=2))

plt.tight_layout()


---
## 7. RSA算法模拟 (Simulating RSA)

下面我们用Python模拟RSA的完整流程。为了演示清楚，我们使用小质数。

### RSA算法步骤

1. 选择两个质数 p 和 q
2. 计算 n = p × q
3. 计算 φ(n) = (p-1)(q-1)
4. 选择公钥指数 e，使得 1 < e < φ(n) 且 gcd(e, φ(n)) = 1
5. 计算私钥指数 d，使得 d × e ≡ 1 (mod φ(n))
6. 公钥 = (e, n)，私钥 = (d, n)
7. 加密：C = M^e mod n
8. 解密：M = C^d mod n

In [ ]:
import math

def simple_rsa_demo():
    """简化版RSA演示"""
    print('='*60)
    print('RSA算法完整模拟 (RSA Algorithm Simulation)')
    print('='*60)
    print()
    
    # Step 1: 选择两个质数
    p = 61
    q = 53
    print(f'Step 1: 选择两个质数')
    print(f'  p = {p}, q = {q}')
    print()
    
    # Step 2: 计算 n
    n = p * q
    print(f'Step 2: 计算 n = p * q')
    print(f'  n = {p} * {q} = {n}')
    print()
    
    # Step 3: 计算欧拉函数
    phi_n = (p - 1) * (q - 1)
    print(f'Step 3: 计算 phi(n) = (p-1)(q-1)')
    print(f'  phi(n) = {p-1} * {q-1} = {phi_n}')
    print()
    
    # Step 4: 选择公钥指数 e
    e = 17  # 常见选择，且 gcd(17, 3120) = 1
    print(f'Step 4: 选择公钥指数 e = {e}')
    print(f'  验证: gcd({e}, {phi_n}) = {math.gcd(e, phi_n)} (必须为1)')
    print()
    
    # Step 5: 计算私钥指数 d
    d = pow(e, -1, phi_n)  # Python 3.8+ 求模逆元
    print(f'Step 5: 计算私钥指数 d')
    print(f'  d = {d}')
    print(f'  验证: d * e mod phi(n) = {d} * {e} mod {phi_n} = {(d * e) % phi_n} (必须为1)')
    print()
    
    # 密钥对
    print(f'公钥 (Public Key):  (e={e}, n={n})')
    print(f'私钥 (Private Key): (d={d}, n={n})')
    print()
    
    # Step 6: 加密
    message = 42  # 要加密的数字 (必须小于n)
    print(f'--- 加密过程 ---')
    print(f'原始消息 M = {message}')
    ciphertext = pow(message, e, n)  # C = M^e mod n
    print(f'密文 C = M^e mod n = {message}^{e} mod {n} = {ciphertext}')
    print()
    
    # Step 7: 解密
    print(f'--- 解密过程 ---')
    decrypted = pow(ciphertext, d, n)  # M = C^d mod n
    print(f'解密 M = C^d mod n = {ciphertext}^{d} mod {n} = {decrypted}')
    print()
    print(f'解密成功! {message} == {decrypted}: {message == decrypted}')



In [ ]:
# 用RSA加密一个完整字符串
import math

def rsa_keygen(p, q):
    """生成RSA密钥对"""
    n = p * q
    phi_n = (p - 1) * (q - 1)
    e = 17
    while math.gcd(e, phi_n) != 1:
        e += 2
    d = pow(e, -1, phi_n)
    return (e, n), (d, n)

def rsa_encrypt_string(message, public_key):
    """逐字符RSA加密"""
    e, n = public_key
    return [pow(ord(ch), e, n) for ch in message]

def rsa_decrypt_string(ciphertext, private_key):
    """逐字符RSA解密"""
    d, n = private_key
    return ''.join([chr(pow(c, d, n)) for c in ciphertext])

# 演示
print('='*60)
print('RSA加密完整字符串演示')
print('='*60)

public_key, private_key = rsa_keygen(61, 53)
print(f'公钥: {public_key}')
print(f'私钥: {private_key}')
print()

msg = 'HELLO'
encrypted = rsa_encrypt_string(msg, public_key)
decrypted = rsa_decrypt_string(encrypted, private_key)

print(f'明文:   {msg}')
print(f'密文:   {encrypted}')
print(f'解密:   {decrypted}')
print()

# 展示每个字符的加密过程
e, n = public_key
print('逐字符加密过程:')
for ch in msg:
    c = pow(ord(ch), e, n)


---
## 8. 对称 vs 非对称加密对比

在实际应用中，两种加密方式经常**配合使用**（如HTTPS）。

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

plt.rcParams['font.sans-serif'] = ['WenQuanYi Zen Hei', 'Noto Sans CJK SC', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False

# 对比表格可视化
fig, ax = plt.subplots(figsize=(14, 8))
ax.axis('off')
ax.set_title('对称加密 vs 非对称加密对比 (Symmetric vs Asymmetric)', fontsize=16, fontweight='bold', pad=20)

# 表格数据
columns = ['特征 / Feature', '对称加密\nSymmetric', '非对称加密\nAsymmetric']
rows = [
    ['密钥数量\nNumber of Keys', '1个共享密钥\n1 Shared Key', '1对(公钥+私钥)\n1 Key Pair'],
    ['速度\nSpeed', '快 (Fast)\n100-1000x', '慢 (Slow)\n1x'],
    ['密钥分发\nKey Distribution', '困难 (Difficult)\n需安全通道', '容易 (Easy)\n公钥可公开'],
    ['典型算法\nAlgorithms', 'AES, DES, 3DES\nBlowfish, RC4', 'RSA, ECC\nDiffie-Hellman'],
    ['典型用途\nUse Cases', '大量数据加密\nBulk Encryption', '密钥交换/数字签名\nKey Exchange/Signing'],
    ['安全性基础\nSecurity Basis', '密钥的保密性\nKey Secrecy', '数学难题\nMath Problems'],
]

# 创建表格
table = ax.table(cellText=rows, colLabels=columns, loc='center',
                  cellLoc='center', colWidths=[0.28, 0.36, 0.36])

# 设置样式
table.auto_set_font_size(False)
table.set_fontsize(10)
table.scale(1, 2.5)

# 表头颜色
for j in range(3):
    cell = table[0, j]
    cell.set_facecolor('#37474F')
    cell.set_text_props(color='white', fontweight='bold', fontsize=11)

# 行颜色交替
for i in range(1, len(rows) + 1):
    color = '#E3F2FD' if i % 2 == 1 else '#FFF3E0'
    for j in range(3):
        table[i, j].set_facecolor(color)
        table[i, j].set_edgecolor('#BDBDBD')

plt.tight_layout()
plt.show()

print('实际应用中的组合策略 (Hybrid Approach):')
print('1. 用非对称加密安全地交换一个临时对称密钥')
print('2. 用这个对称密钥加密实际的数据传输')
print('3. 这样既解决了密钥分发问题，又保证了加密速度')


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import time

plt.rcParams['font.sans-serif'] = ['WenQuanYi Zen Hei', 'Noto Sans CJK SC', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False

# 性能对比：模拟对称 vs 非对称加密速度
def simulate_symmetric(data_size):
    """模拟对称加密(XOR)"""
    data = bytearray(np.random.randint(0, 256, data_size, dtype=np.uint8))
    key = bytearray(np.random.randint(0, 256, 32, dtype=np.uint8))
    start = time.perf_counter()
    result = bytearray(len(data))
    for i in range(len(data)):
        result[i] = data[i] ^ key[i % len(key)]
    return time.perf_counter() - start

def simulate_asymmetric(data_size):
    """模拟非对称加密(模幂运算)"""
    e, n = 17, 3233
    data = [np.random.randint(32, 126) for _ in range(data_size)]
    start = time.perf_counter()
    result = [pow(d, e, n) for d in data]
    return time.perf_counter() - start

sizes = [100, 500, 1000, 2000, 5000]
sym_times = [simulate_symmetric(s) * 1000 for s in sizes]
asym_times = [simulate_asymmetric(s) * 1000 for s in sizes]

fig, ax = plt.subplots(figsize=(12, 6))
x = np.arange(len(sizes))
width = 0.35

bars1 = ax.bar(x - width/2, sym_times, width, label='Symmetric (XOR)', color='#4CAF50', edgecolor='black')
bars2 = ax.bar(x + width/2, asym_times, width, label='Asymmetric (Modular Exp)', color='#F44336', edgecolor='black')

ax.set_xlabel('数据大小 (字节)', fontsize=12)
ax.set_ylabel('时间 (毫秒)', fontsize=12)
ax.set_title('对称加密 vs 非对称加密 速度对比', fontsize=14, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels([str(s) for s in sizes])
ax.legend(fontsize=12)
ax.grid(axis='y', alpha=0.3)

# 添加数值标签
for bar in bars1:
    h = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2., h, f'{h:.1f}',
            ha='center', va='bottom', fontsize=9)
for bar in bars2:
    h = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2., h, f'{h:.1f}',
            ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.show()

print('结论: 非对称加密比对称加密慢得多!')


---
## 9. 知识总结

### 核心考点 (Key Exam Points)

1. **对称加密**使用同一把密钥进行加密和解密，速度快但有密钥分发问题
2. **非对称加密**使用公钥/私钥对，解决了密钥分发问题但速度慢
3. **XOR**是加密的基础运算，因为它是可逆的且能隐藏数据模式
4. **RSA**基于大质数分解的计算困难性
5. 实际应用中常采用**混合加密**（Hybrid Encryption）
6. AES是当前对称加密标准，RSA是经典非对称加密算法

---
## 10. 练习题 (Exercises)

### 练习1：凯撒密码
用凯撒密码（shift=5）加密 `"COMPUTER SCIENCE"` 并写出密文。然后写代码验证。

### 练习2：XOR加密
手动计算：明文 `10110011` 与密钥 `01101010` 的XOR加密结果是什么？再XOR一次密钥验证还原。

### 练习3：RSA概念题
Alice想给Bob发送加密消息。请回答：
- (a) Alice应该用谁的公钥加密？
- (b) Bob用什么来解密？
- (c) 如果Eve截获了Bob的公钥和密文，她能解密吗？为什么？

### 练习4：对比分析
解释为什么HTTPS协议同时使用对称加密和非对称加密，而不是只用其中一种？

### 练习5：编程挑战
修改下面的XOR加密函数，使其支持中文字符加密。

In [ ]:
# 练习1 验证区域
# 请在这里写你的代码

# encrypted = caesar_encrypt('COMPUTER SCIENCE', 5)


In [ ]:
# 练习2 验证区域
# 请在这里手动计算后用代码验证

# plaintext = 0b10110011
# key = 0b01101010
# ciphertext = plaintext ^ key
# restored = ciphertext ^ key
# print(f'密文: {ciphertext:08b}')


In [ ]:
# 练习5 编程挑战
# 提示：中文UTF-8编码可能有多个字节
# 你需要对字节序列进行XOR，而不是对字符

def xor_encrypt_chinese(message, key):
    """支持中文的XOR加密"""
    # 请在这里实现你的代码
    pass

# 测试
# result = xor_encrypt_chinese('你好世界', 'KEY')
